# Target versus realized tetrahedron edge size

Mesh generators do not always realize the requested edge size exactly. This notebook converts the same small particle with a few physical target edge lengths, then prints and plots the actual tetrahedron edge-length statistics in both native STL units and meters.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path.cwd()
if not (REPO / "src").exists() and (REPO.parent / "src").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

from stl2fem.conversion import tetrahedralize_stl_for_merrill
from stl2fem.datasets import assign_size_bins, nikolaisen_inventory
from stl2fem.memory import estimate_merrill_memory
from stl2fem.quality import inspect_surface_stl, inspect_volume_mesh, load_surface, load_volume_mesh
from stl2fem.units import DEFAULT_TARGET_EDGE_LENGTH_M, add_meter_scaled_columns, make_unit_context

DATASET_ROOT = REPO / "data" / "Nikolaisen2022"
OUTPUT_ROOT = REPO / "processed" / "demo"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SAMPLE_STL = DATASET_ROOT / "Plag Binary meshes" / "PLAG246-binary.stl"
INPUT_UNIT = "um"
TARGET_EDGE_LENGTH_M = DEFAULT_TARGET_EDGE_LENGTH_M

The default is `9e-9 m`. Because the sample STL is interpreted as micrometers, this is `0.009` native coordinate units. The coarser targets below are included to make the contrast obvious and keep the demo fast.

In [ ]:
targets_m = [5e-8, 2e-8, DEFAULT_TARGET_EDGE_LENGTH_M]
rows = []

units = make_unit_context(input_unit=INPUT_UNIT, target_edge_length_m=DEFAULT_TARGET_EDGE_LENGTH_M)
for target_m in targets_m:
    current_units = make_unit_context(input_unit=INPUT_UNIT, target_edge_length_m=target_m)
    native_msh = OUTPUT_ROOT / f"PLAG246_target_{target_m:.0e}_native.msh"
    merrill_msh = OUTPUT_ROOT / f"PLAG246_target_{target_m:.0e}_meters.msh"
    tetrahedralize_stl_for_merrill(
        SAMPLE_STL,
        native_msh,
        merrill_msh,
        input_unit=INPUT_UNIT,
        target_edge_length_m=target_m,
        overwrite=True,
    )
    quality = add_meter_scaled_columns(
        inspect_volume_mesh(native_msh),
        input_scale_to_meters=current_units.input_scale_to_meters,
    )
    memory = estimate_merrill_memory(quality["n_nodes"], quality["n_tets"])
    rows.append({
        "target_edge_length_m": target_m,
        "target_edge_length_native": current_units.target_edge_length_native,
        **quality,
        **memory,
    })

results = pd.DataFrame(rows)
columns = [
    "target_edge_length_m", "target_edge_length_native",
    "edge_length_median", "edge_length_p95", "edge_length_median_m", "edge_length_p95_m",
    "n_nodes", "n_tets", "msh_size_mib", "estimated_memory_human",
]
results[columns]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(results["target_edge_length_m"], results["edge_length_median_m"], marker="o", label="median realized edge")
ax.plot(results["target_edge_length_m"], results["edge_length_p95_m"], marker="o", label="p95 realized edge")
ax.plot(results["target_edge_length_m"], results["target_edge_length_m"], linestyle="--", color="black", label="requested target")
ax.invert_xaxis()
ax.set_xlabel("Requested target edge length [m]")
ax.set_ylabel("Realized tetra edge length [m]")
ax.set_title("Requested vs realized tetrahedron edge size")
ax.legend()
plt.show()

For the full particle set, use the size-bin notebooks. They print the same realized edge-length columns for every converted mesh and cap PyVista display output with `DISPLAY_LIMIT`.